<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
This notebook adapts material from the <a href="https://docs.ros.org/en/jazzy/">ROS 2 documentation</a> (<a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a>) and from the Duckietown <em>ROS Basics</em> and <em>ROS 2 Middleware</em> learning experiences.
</span>

# Build and Discover a ROS 2 Package

The previous notebook located the source files of `ros2_basics` and traced their executable names to Python functions. Those installation instructions now need to be applied so that ROS 2 can find the package and its executables.

After changing a node, the terminal needs to find the updated package. A workspace build prepares its installed files, and sourcing the workspace overlay makes them discoverable to ROS 2.

This notebook builds `ros2_basics`, loads the resulting workspace environment, inspects the installed package and runs its first node. The observations connect three steps: discovering source, preparing an installation, and making that installation available to ROS 2 commands.

## Prepare the terminal

The commands run in one terminal, starting from the root of this repository. Save the repository location and move to the workspace root:

```bash
cd /code/src/lx-ros2-basics
LX_DIR="$PWD"
cd "$COLCON_WS"
```

`LX_DIR` preserves the repository path, so that source files can still be located after changing directories. In the editor, the repository is `/code/src/lx-ros2-basics` and the workspace root is `/code`.

The source-discovery command of the previous notebook can now use that saved path:

```bash
colcon list --base-paths "$LX_DIR/packages"
```

```shell
ros2_basics	/code/src/lx-ros2-basics/packages/ros2_basics	(ros.ament_python)
```

The path (the middle output) identifies the source that will be built. Keeping discovery scoped to this repository avoids accidentally including another copy of the package elsewhere in the workspace. Before building, check whether the terminal already knows of an installed version:

```bash
ros2 pkg prefix ros2_basics
```

The expected response at the first execution of this notebook is:

```shell
Package not found
```

That result is expected before the package has been installed and made available to the terminal. If a path appears instead, an existing installation is already visible; its location can be compared with the result after the build.

## Build the supplied package

The build uses the installation instructions inspected in the previous notebook to prepare the Python code, package metadata, launch files, configuration and executable entry points. From the workspace root:

```bash
colcon build --base-paths "$LX_DIR/packages" --packages-select ros2_basics
```

`--base-paths` identifies where source discovery begins, while `--packages-select` chooses the package to build from those discovered there. The build should finish with output such as:

```shell
Starting >>> ros2_basics
Finished <<< ros2_basics [1.2s]

Summary: 1 package finished [1.4s]
```

The `Finished` line and the summary report that the package completed successfully (execution times vary). If the summary mentions `stderr output`, read it: warnings about disabled byte-compilation (`byte-compiling is disabled, skipping`) are harmless; a `SyntaxError` in one of the node files is not, and the package is then not installed.

The build produces its working files under `/code/build`, its installed files under `/code/install`, and its diagnostic records under `/code/log`. For this package, the installation is placed in `/code/install/ros2_basics`. Application changes belong in the source repository, where they can be incorporated by a subsequent build.

**Note:** `dts code build -R ROBOTNAME`, run from your laptop, performs the same `colcon build` of your `packages/` inside the Docker image used by `dts code workbench`. The editor build above is what you use while developing; the `dts` build is what runs on the robot.

## Make the installation available to the terminal

Although a completed build leaves files on disk, the current shell also needs the search paths that allow ROS 2 to locate them. Repeating the package lookup after the build and before loading the overlay still reports that the package cannot be found:

```bash
ros2 pkg prefix ros2_basics
```

```shell
Package not found
```

The workspace provides a setup script that adds its installed packages to the current shell's environment (successful sourcing normally produces no output):

```bash
source install/setup.bash
```

The installed ROS 2 distribution supplies the underlying libraries and tools, while the workspace adds the exercise package on top. In this arrangement, the distribution is the **underlay** and the workspace is an **overlay**. Sourcing applies the overlay's environment settings to the shell executing the command, allowing subsequent ROS 2 commands in that shell to find its packages. Check `AMENT_PREFIX_PATH` again: `/code/install/ros2_basics` is now in front.

## Confirm package discovery

With the overlay loaded, the same lookup should resolve the installed package:

```bash
ros2 pkg prefix ros2_basics
```

```shell
/code/install/ros2_basics
```

This is the installation prepared by the build. Checking the prefix is particularly useful when several workspaces contain a package with the same name, because it identifies the installation selected by the current terminal.

The next check connects that installation to the executable registrations examined in notebook 05:

```bash
ros2 pkg executables ros2_basics
```

All three supplied executables should appear:

```shell
ros2_basics drive_node
ros2_basics my_publisher_node
ros2_basics my_subscriber_node
```

Each line contains a package name followed by an executable name. These entries provide the package and executable names needed by `ros2 run`. The launch files and configuration are installed too:

```bash
ls "$(ros2 pkg prefix ros2_basics)/share/ros2_basics/launch"
ls "$(ros2 pkg prefix ros2_basics)/share/ros2_basics/config/drive_node"
```

## Run the first node

`ros2 run` starts an executable of an installed package:

```bash
ros2 run ros2_basics my_publisher_node
```

The node should begin logging:

```shell
[INFO] [1793450000.123456789] [my_publisher_node]: publishing on '/chatter' at 1.0 Hz
[INFO] [1793450001.123456789] [my_publisher_node]: publishing: "Hello from ROS 2! [0]"
[INFO] [1793450002.123456789] [my_publisher_node]: publishing: "Hello from ROS 2! [1]"
```

Each line shows the log level, a time stamp, the node name and the message. The executable name was `my_publisher_node`; the node name shown in the log also happens to be `my_publisher_node`, because that is what the Python code chose. Stop it with `Ctrl-C` for now; the next notebook inspects it while it runs.

If the command fails with `Error opening session: Unable to connect to any of [tcp/...:7447]`, no router is reachable: re-read the "Before you start" section of notebook 03.

## Use the package from another terminal

Each terminal has its own ROS 2 environment, so a terminal opened later also needs access to the workspace installation. Open a new terminal and run:

```bash
ros2 pkg prefix ros2_basics
```

In this editor, new terminals automatically source `/opt/ros/jazzy/setup.bash` and, when it exists, `$COLCON_WS/install/setup.bash`, so the answer should already be `/code/install/ros2_basics`. In an environment without that convenience, the explicit commands are:

```bash
source /opt/ros/jazzy/setup.bash
source "$COLCON_WS/install/setup.bash"
```

Terminals that were already open before the first build need the second command once. A newly opened terminal can use the existing installation without rebuilding it. Rebuilding becomes necessary when source or package configuration changes need to be incorporated into the installed version; the terminal that runs the node then needs a restart of that node (and, if the terminal predates the first build, a `source` of the overlay).

## Recover package discovery

If `ros2 run` cannot find the package (`Package 'ros2_basics' not found`) or the executable, check the build and the current terminal's overlay in this order:

1. `cd "$COLCON_WS"`;
2. build with `colcon build --base-paths "$LX_DIR/packages" --packages-select ros2_basics` (with `LX_DIR=/code/src/lx-ros2-basics`);
3. `source install/setup.bash` in the terminal that will run the command;
4. confirm with `ros2 pkg prefix ros2_basics` and `ros2 pkg executables ros2_basics`.

A build error points to package preparation (read the error in the build output, or in `/code/log/latest_build/ros2_basics/`); a successful build followed by discovery after sourcing points to the terminal's environment. This order helps locate the problem before changing node code.

Building applied the package's installation instructions, and sourcing the overlay made the resulting installation available to the terminal. Package-prefix and executable queries then identified the selected installation and its registered programs, and `ros2 run` started one of them. Graph inspection can now connect the running publisher to its topic, message rate and received data.

## Further reading

The [colcon quick-start guide](https://colcon.readthedocs.io/en/released/user/quick-start.html) describes package selection, workspace builds and loading the resulting environment.

Let's move to [the next notebook, where we inspect the running node](../notebooks/07_inspect_a_local_graph.ipynb).